# English-Hindi Audio Deepfake Detection

## Publication-Ready Training and Evaluation Notebook

**Project topic:** Multi Audio Deepfake Audio Detection in English and Hindi  
**Final protocol:** `RawCh` held-out-generator challenge split  
**Final model:** `SafariWavLMEnsemble` with WavLM embeddings and 5-seed probability ensemble

This notebook is structured for professor review. It explains the dataset, model architecture, training method, evaluation protocol, and final visual results in one run.

### What This Notebook Does

1. Loads the English-Hindi challenge dataset.
2. Shows dataset split, class distribution, and language distribution charts.
3. Trains the model on five random seeds.
4. Generates validation and test predictions.
5. Ensembles the five models.
6. Selects the threshold using validation data only.
7. Evaluates on the held-out test set.
8. Generates ROC curves, loss curves, F1 curves, confusion matrix, and final dashboard.

### Target Result

The previous completed run achieved:

| Metric | Test Result |
|---|---:|
| ROC-AUC | 0.9668 |
| Accuracy | 0.9048 |
| F1 Score | 0.8993 |

Running all cells should reproduce the same experiment structure. Results may vary slightly if the model is retrained.

## 1. Environment Setup

This cell imports all required packages and automatically detects whether the notebook is being run from the full project folder or from the packaged `SDP_Model` handoff folder.

In [ ]:
from pathlib import Path
from argparse import Namespace
import csv
import json
import sys
import subprocess

import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

ROOT = Path.cwd().resolve()
if not (ROOT / 'safari_wavlm_ensemble_train.py').exists() and not (ROOT / 'src' / 'safari_wavlm_ensemble_train.py').exists():
    ROOT = ROOT.parent
CODE_ROOT = ROOT / 'src' if (ROOT / 'src' / 'safari_wavlm_ensemble_train.py').exists() else ROOT
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from safari_wavlm_ensemble_train import train as train_safari_wavlm
from predict_safari_wavlm_ensemble import device_from_arg, load_ckpt, build_model, run_batch
from generate_publication_graphs import (
    set_style,
    plot_dataset_split_bar,
    plot_class_distribution,
    plot_language_distribution,
)

set_style()
print('Project root:', ROOT)
print('Code root:', CODE_ROOT)
print('Python:', sys.executable)
print('CUDA available:', torch.cuda.is_available())

## 2. Experiment Configuration

For professor review, the notebook is configured to **train the full five-seed model by default**.

If you only want to regenerate graphs from already trained artifacts, change:

```python
RUN_TRAINING = False
RUN_PREDICTION = False
```

In [ ]:
DATA_ROOT = ROOT / 'datasets' if (ROOT / 'datasets').exists() else ROOT

TRAIN_CSV = DATA_ROOT / 'RawCh_train' / 'RawCh_train' / 'balanced_index.csv'
VAL_CSV = DATA_ROOT / 'RawCh_val' / 'RawCh_val' / 'balanced_index.csv'
TEST_CSV = DATA_ROOT / 'RawCh_test' / 'RawCh_test' / 'balanced_index.csv'
FEATURE_ROOT = DATA_ROOT / 'WavLM_embeddings_unified'

ARTIFACT_ROOT = ROOT / 'artifacts' / 'publication_challenge_improved'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
GRAPH_DIR = ARTIFACT_ROOT / 'graphs'
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [17, 42, 77, 123, 202]
CURRENT_SEED = 17

# Professor-review default: run complete training and prediction.
RUN_TRAINING = True
RUN_PREDICTION = True
RUN_ALL_SEEDS = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = torch.cuda.is_available()
EPOCHS = 60
BATCH_SIZE = 512
NUM_WORKERS = 4

print('Data root:', DATA_ROOT)
print('Feature root:', FEATURE_ROOT)
print('Artifact root:', ARTIFACT_ROOT)
print('Graph root:', GRAPH_DIR)
print('Device:', DEVICE, '| AMP:', AMP)
print('Seeds:', SEEDS)

## 3. Dataset Used

The final model uses the **RawCh held-out-generator challenge dataset**, not the older `Balanced_*` dataset.

### Why RawCh?

`RawCh` tests whether the detector can recognize fake speech from generators that were not seen during training.

| Split | English Fake Generators |
|---|---|
| Train | FlashSpeech, NaturalSpeech3, PromptTTS2, VALLE, VoiceBox |
| Validation | SeedTTS |
| Test | OpenAI, xTTS |

This is more credible than a random split because it better represents real-world deepfake detection.

In [ ]:
def read_df(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def source_from_audio_path(value: str) -> str:
    parts = Path(str(value)).parts
    if 'Raw Dataset' in parts:
        i = parts.index('Raw Dataset')
        return '/'.join(parts[i + 1:i + 3])
    return '<unknown>'


def summarize_split(name: str, csv_path: Path, feature_root: Path) -> pd.DataFrame:
    df = read_df(csv_path)
    print(f'[{name}] rows={len(df)} columns={list(df.columns)}')
    print('label counts:', df['label'].value_counts().to_dict())
    print('language counts:', df['language'].value_counts().to_dict())
    print('language x label:')
    print(pd.crosstab(df['language'], df['label']))
    missing = [p for p in df['feature_path'] if not (feature_root / p).exists()]
    print('missing feature files:', len(missing))
    if 'audio_path' in df.columns:
        print('source counts:')
        print(df['audio_path'].map(source_from_audio_path).value_counts())
    print()
    return df

train_df = summarize_split('train', TRAIN_CSV, FEATURE_ROOT)
val_df = summarize_split('validation', VAL_CSV, FEATURE_ROOT)
test_df = summarize_split('test', TEST_CSV, FEATURE_ROOT)

sample_feature = FEATURE_ROOT / train_df.iloc[0]['feature_path']
x = np.load(sample_feature)
print('sample feature:', sample_feature)
print('sample shape:', x.shape, 'mean:', float(x.mean()), 'std:', float(x.std()))

## 4. Dataset Visualizations

These figures show the dataset size, real/fake distribution, and English/Hindi distribution.

In [ ]:
dataset_tables = {
    'Train': train_df,
    'Validation': val_df,
    'Test': test_df,
}

plot_dataset_split_bar(dataset_tables, GRAPH_DIR / '09_dataset_split_bar_chart.png')
plot_class_distribution(dataset_tables, GRAPH_DIR / '10_class_distribution_chart.png')
plot_language_distribution(dataset_tables, GRAPH_DIR / '11_language_distribution_chart.png')

for image_name in [
    '09_dataset_split_bar_chart.png',
    '10_class_distribution_chart.png',
    '11_language_distribution_chart.png',
]:
    print(image_name)
    display(Image(filename=str(GRAPH_DIR / image_name)))

## 5. Model Architecture

Final model: **SafariWavLMEnsemble**

The model uses precomputed WavLM embeddings as input. Each sample is represented as a 768-dimensional `.npy` vector.

### Main Components

| Component | Role |
|---|---|
| WavLM embeddings | Speech representation extracted from raw audio |
| Safari branch | AFUM fusion + Transformer reasoning over augmented feature views |
| WavLM MLP branch | Learns deepfake-specific representation from WavLM vector |
| Language embedding | Conditions model on English/Hindi language code |
| Fusion classifier | Produces final fake probability |

The following figure summarizes the full input-output flow.

In [ ]:
# This figure is generated later by the graph script too. If it already exists, display it now.
pipeline_path = GRAPH_DIR / '12_input_output_pipeline_figure.png'
if pipeline_path.exists():
    display(Image(filename=str(pipeline_path)))
else:
    print('Pipeline figure will be generated after evaluation graph generation.')

## 6. Training Setup

This cell trains one model for each seed. The five models are later ensembled by averaging fake probabilities.

### Training Details

| Parameter | Value |
|---|---:|
| Epochs | 60 |
| Batch size | 512 |
| Hidden dimension | 384 |
| Language embedding dimension | 32 |
| Optimizer | AdamW |
| LR | 3e-4 |
| Max LR | 2e-3 |
| Weight decay | 1e-4 |
| Early stopping patience | 8 |
| Seeds | 17, 42, 77, 123, 202 |

In [ ]:
def seed_dir(seed: int) -> Path:
    d = ARTIFACT_ROOT / f'seed_{seed}'
    d.mkdir(parents=True, exist_ok=True)
    return d


def make_train_args(seed: int) -> Namespace:
    return Namespace(
        project_root=str(ROOT),
        train_split='RawCh_train',
        val_split='RawCh_val',
        test_split='RawCh_test',
        train_csv=str(TRAIN_CSV),
        val_csv=str(VAL_CSV),
        test_csv=str(TEST_CSV),
        train_feature_dir=str(FEATURE_ROOT),
        val_feature_dir=str(FEATURE_ROOT),
        test_feature_dir=str(FEATURE_ROOT),
        wavlm_train_feature_dir=str(FEATURE_ROOT),
        wavlm_val_feature_dir=str(FEATURE_ROOT),
        wavlm_test_feature_dir=str(FEATURE_ROOT),
        out_dir=str(seed_dir(seed)),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        seed=seed,
        hidden_dim=384,
        language_emb_dim=32,
        lr=3e-4,
        max_lr=2e-3,
        weight_decay=1e-4,
        patience=8,
        view_noise_std=0.01,
        view_drop_prob=0.03,
        stats_samples=50000,
        device=DEVICE,
        amp=AMP,
    )


def train_one_seed(seed: int) -> Path:
    print(f'\n========== Training seed {seed} ==========')
    train_safari_wavlm(make_train_args(seed))
    return seed_dir(seed) / 'best_safari_wavlm_ensemble.pt'

if RUN_TRAINING:
    seeds_to_run = SEEDS if RUN_ALL_SEEDS else [CURRENT_SEED]
    for seed in seeds_to_run:
        train_one_seed(seed)
else:
    print('Training skipped. Existing seed checkpoints will be used if available.')

## 7. Prediction Generation

After training, this cell generates validation and test predictions for each seed model.

In [ ]:
def predict_seed_split(seed: int, split: str) -> Path:
    d = seed_dir(seed)
    model_path = d / 'best_safari_wavlm_ensemble.pt'
    if not model_path.exists():
        raise FileNotFoundError(f'Missing checkpoint for seed {seed}: {model_path}')

    input_csv = {'val': VAL_CSV, 'test': TEST_CSV}[split]
    output_csv = d / f'{split}_predictions.csv'
    device_obj = device_from_arg(DEVICE)
    ckpt = load_ckpt(model_path, device_obj)
    model = build_model(ckpt, device_obj)
    run_batch(
        model=model,
        ckpt=ckpt,
        input_csv=input_csv,
        base_feature_root=FEATURE_ROOT,
        wavlm_feature_root=FEATURE_ROOT,
        output_csv=output_csv,
        device=device_obj,
    )
    return output_csv

if RUN_PREDICTION:
    seeds_to_predict = SEEDS if RUN_ALL_SEEDS else [CURRENT_SEED]
    for seed in seeds_to_predict:
        print(f'Predicting validation and test for seed {seed}')
        predict_seed_split(seed, 'val')
        predict_seed_split(seed, 'test')
else:
    print('Prediction skipped. Existing prediction CSVs will be used if available.')

## 8. Ensemble and Threshold Selection

The final prediction is produced by averaging fake probabilities from the five seed models.

The classification threshold is selected only on the validation set, then applied unchanged to the test set.

In [ ]:
def completed_seeds() -> list[int]:
    done = []
    for seed in SEEDS:
        d = seed_dir(seed)
        if (d / 'val_predictions.csv').exists() and (d / 'test_predictions.csv').exists():
            done.append(seed)
    return done


def read_predictions(path: Path) -> dict[str, dict]:
    with path.open('r', encoding='utf-8', newline='') as handle:
        return {row['feature_path']: row for row in csv.DictReader(handle)}


def aggregate_seed_predictions(seeds: list[int], split: str) -> pd.DataFrame:
    tables = []
    for seed in seeds:
        path = seed_dir(seed) / f'{split}_predictions.csv'
        tables.append(read_predictions(path))
    common_keys = sorted(set.intersection(*(set(t.keys()) for t in tables)))
    rows = []
    for key in common_keys:
        base = dict(tables[0][key])
        probs = [float(t[key]['fake_probability']) for t in tables]
        base['fake_probability'] = float(np.mean(probs))
        rows.append(base)
    return pd.DataFrame(rows)


done = completed_seeds()
print('Completed seeds:', done)
if not done:
    raise RuntimeError('No completed predictions found. Train/predict at least one seed first.')

val_ens = aggregate_seed_predictions(done, 'val')
test_ens = aggregate_seed_predictions(done, 'test')
print('Validation rows:', len(val_ens), '| Test rows:', len(test_ens))
val_ens.head()

## 9. Final Metrics

This section computes validation and test performance using the validation-selected threshold.

In [ ]:
def to_binary_label(series: pd.Series) -> np.ndarray:
    return series.astype(str).str.lower().eq('fake').astype(int).to_numpy()


def threshold_candidates(scores: np.ndarray) -> np.ndarray:
    return np.unique(
        np.concatenate([
            np.linspace(0.0, 1.0, 1001),
            np.quantile(scores, np.linspace(0.0, 1.0, 1001)),
        ])
    )


def compute_metrics(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    pred = (scores >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, average='binary', zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        'n': int(len(y_true)),
        'threshold': float(threshold),
        'roc_auc': float(roc_auc_score(y_true, scores)) if len(set(y_true)) > 1 else float('nan'),
        'accuracy': float(accuracy_score(y_true, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, pred)),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }


def find_best_threshold(y_true: np.ndarray, scores: np.ndarray) -> tuple[float, dict]:
    best = (-1.0, -1.0, 0.5)
    best_metrics = {}
    for t in threshold_candidates(scores):
        pred = (scores >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        acc = accuracy_score(y_true, pred)
        if (f1, acc) > (best[0], best[1]):
            best = (float(f1), float(acc), float(t))
            best_metrics = compute_metrics(y_true, scores, float(t))
    return best[2], best_metrics


def evaluate_table(df: pd.DataFrame, threshold: float) -> dict:
    y = to_binary_label(df['label'])
    scores = df['fake_probability'].astype(float).to_numpy()
    return compute_metrics(y, scores, threshold)


val_y = to_binary_label(val_ens['label'])
val_scores = val_ens['fake_probability'].astype(float).to_numpy()
best_t, val_best_metrics = find_best_threshold(val_y, val_scores)

test_metrics = evaluate_table(test_ens, best_t)
print('Best validation threshold:', best_t)
print('Validation metrics:')
print(json.dumps(val_best_metrics, indent=2))
print('Test metrics:')
print(json.dumps(test_metrics, indent=2))

## 10. Per-Language and Per-Source Metrics

These tables show whether the model performs consistently across English, Hindi, and individual source/generator groups.

In [ ]:
def evaluate_single_class(df: pd.DataFrame, threshold: float) -> dict:
    y = to_binary_label(df['label'])
    scores = df['fake_probability'].astype(float).to_numpy()
    pred = (scores >= threshold).astype(int)
    return {
        'n': int(len(df)),
        'threshold': float(threshold),
        'roc_auc': float('nan'),
        'accuracy': float(accuracy_score(y, pred)),
        'balanced_accuracy': float('nan'),
        'precision': float('nan'),
        'recall': float('nan'),
        'f1': float('nan'),
        'tn': int(((y == 0) & (pred == 0)).sum()),
        'fp': int(((y == 0) & (pred == 1)).sum()),
        'fn': int(((y == 1) & (pred == 0)).sum()),
        'tp': int(((y == 1) & (pred == 1)).sum()),
        'mean_fake_probability': float(scores.mean()),
    }


def grouped_metrics(df: pd.DataFrame, threshold: float, group_col: str) -> pd.DataFrame:
    rows = []
    for name, sub in df.groupby(group_col):
        y = to_binary_label(sub['label'])
        scores = sub['fake_probability'].astype(float).to_numpy()
        m = compute_metrics(y, scores, threshold) if len(set(y)) > 1 else evaluate_single_class(sub, threshold)
        m[group_col] = name
        rows.append(m)
    return pd.DataFrame(rows).sort_values('n', ascending=False)


print('Per-language test metrics')
display(grouped_metrics(test_ens, best_t, 'language'))

if 'audio_path' in test_ens.columns:
    test_with_source = test_ens.copy()
    test_with_source['source'] = test_with_source['audio_path'].map(source_from_audio_path)
    print('Per-source test metrics')
    display(grouped_metrics(test_with_source, best_t, 'source'))

## 11. Save Final Predictions and Summary

This cell saves final ensemble predictions and the final summary JSON used by the graph generator.

In [ ]:
def add_predictions(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    out = df.copy()
    out['fake_probability'] = out['fake_probability'].astype(float)
    out['predicted_label'] = np.where(out['fake_probability'] >= threshold, 'fake', 'real')
    return out

out_dir = ARTIFACT_ROOT / 'ensemble_predictions'
out_dir.mkdir(parents=True, exist_ok=True)
val_out = out_dir / 'val_improved_predictions.csv'
test_out = out_dir / 'test_improved_predictions.csv'

val_saved = add_predictions(val_ens, best_t)
test_saved = add_predictions(test_ens, best_t)
val_saved.to_csv(val_out, index=False)
test_saved.to_csv(test_out, index=False)

summary = {
    'protocol': 'RawCh held-out-generator challenge',
    'model': 'SafariWavLMEnsemble',
    'feature_root': str(FEATURE_ROOT),
    'seeds': done,
    'threshold': float(best_t),
    'val_metrics': evaluate_table(val_saved, best_t),
    'test_metrics': evaluate_table(test_saved, best_t),
    'val_predictions': str(val_out),
    'test_predictions': str(test_out),
}

summary_path = ARTIFACT_ROOT / 'publication_challenge_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Saved:', summary_path)
print(json.dumps(summary, indent=2))

## 12. Generate All Graphs and Matrices

This cell regenerates all publication figures from the final saved metrics and predictions.

In [ ]:
graph_script = CODE_ROOT / 'generate_publication_graphs.py'
if not graph_script.exists():
    raise FileNotFoundError(f'Missing graph generator: {graph_script}')

cmd = [
    sys.executable,
    str(graph_script),
    '--artifact-dir',
    str(ARTIFACT_ROOT),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(ROOT), check=True)
print('Graphs ready at:', GRAPH_DIR)

## 13. Training and Validation Graphs

These figures show the model's learning behavior across epochs and seeds.

In [ ]:
for image_name in [
    '01_validation_roc_auc_per_epoch.png',
    '02_loss_per_epoch.png',
    '03_validation_f1_per_epoch.png',
    '04_validation_accuracy_by_seed.png',
]:
    print(image_name)
    display(Image(filename=str(GRAPH_DIR / image_name)))

## 14. Final Test Evaluation Graphs

These figures show final test-set performance.

In [ ]:
for image_name in [
    '05_per_class_metrics_test_set.png',
    '06_roc_curve_test_set.png',
    '07_confusion_matrix_test_set.png',
    '08_evaluation_dashboard.png',
]:
    print(image_name)
    display(Image(filename=str(GRAPH_DIR / image_name)))

## 15. Input-Output Pipeline Figure

This final diagram explains the full model pipeline.

In [ ]:
print('12_input_output_pipeline_figure.png')
display(Image(filename=str(GRAPH_DIR / '12_input_output_pipeline_figure.png')))

## 16. Final Observation for Review

The final five-seed ensemble should be interpreted as follows:

- The model crosses the project target of **95%+ ROC-AUC** and **90%+ accuracy** on the held-out-generator challenge test set.
- The evaluation is stronger than a random split because the English fake generators in the test set, OpenAI and xTTS, are not used during training.
- Hindi performance is very high, but this should be discussed carefully because Hindi fake data appears less generator-diverse.
- The main remaining weakness is OpenAI English fake detection.

This notebook therefore supports a credible project result, while still identifying limitations honestly for research discussion.